# Multi-Head Attention (多头注意力) 实现

多头注意力是Transformer的核心组件，通过多个注意力头并行计算，捕获不同子空间的信息。

**公式**：
$$
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O
$$
$$
\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')

## 1. 辅助函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Query矩阵 (seq_len, d_k)
        K: Key矩阵 (seq_len, d_k)
        V: Value矩阵 (seq_len, d_v)
        mask: 注意力掩码
    
    Returns:
        output: 输出 (seq_len, d_v)
        attention_weights: 注意力权重 (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # 计算注意力得分
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # 应用mask
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Softmax
    attention_weights = softmax(scores, axis=-1)
    
    # 加权求和
    output = np.dot(attention_weights, V)
    
    return output, attention_weights

## 2. 实现多头注意力类

In [ ]:
class MultiHeadAttention:
    """
    多头注意力机制
    
    通过多个注意力头并行计算，捕获不同子空间的信息
    """
    
    def __init__(self, embed_dim, num_heads, use_bias=True):
        """
        初始化多头注意力层
        
        Args:
            embed_dim: 嵌入维度（必须能被num_heads整除）
            num_heads: 注意力头的数量
            use_bias: 是否使用偏置
        """
        assert embed_dim % num_heads == 0, "embed_dim必须能被num_heads整除"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.use_bias = use_bias
        
        # Q、K、V投影矩阵
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        # 输出投影矩阵
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(embed_dim)
            self.b_v = np.zeros(embed_dim)
            self.b_o = np.zeros(embed_dim)
    
    def split_heads(self, x):
        """
        将输入分割成多个头
        
        (seq_len, embed_dim) -> (num_heads, seq_len, head_dim)
        """
        seq_len = x.shape[0]
        # 重塑并转置
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 0, 2)
        return x
    
    def combine_heads(self, x):
        """
        将多个头的输出合并
        
        (num_heads, seq_len, head_dim) -> (seq_len, embed_dim)
        """
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.embed_dim)
        return x
    
    def forward(self, query, key=None, value=None, mask=None, return_attention=False):
        """
        前向传播
        
        Args:
            query: Query输入 (seq_len_q, embed_dim)
            key: Key输入 (seq_len_k, embed_dim)，None则使用query
            value: Value输入 (seq_len_v, embed_dim)，None则使用key
            mask: 注意力掩码
            return_attention: 是否返回注意力权重
        
        Returns:
            output: 输出 (seq_len_q, embed_dim)
            attention_weights: (可选) 注意力权重
        """
        # 自注意力：key和value使用query
        if key is None:
            key = query
        if value is None:
            value = key
        
        # 1. 线性投影
        Q = np.dot(query, self.W_q)
        K = np.dot(key, self.W_k)
        V = np.dot(value, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # 2. 分割成多个头
        Q = self.split_heads(Q)  # (num_heads, seq_len_q, head_dim)
        K = self.split_heads(K)  # (num_heads, seq_len_k, head_dim)
        V = self.split_heads(V)  # (num_heads, seq_len_v, head_dim)
        
        # 3. 对每个头计算attention
        head_outputs = []
        attention_weights_list = []
        
        for i in range(self.num_heads):
            head_output, attn_weights = scaled_dot_product_attention(
                Q[i], K[i], V[i], mask=mask
            )
            head_outputs.append(head_output)
            attention_weights_list.append(attn_weights)
        
        # 4. 合并头
        multi_head_output = np.stack(head_outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        
        # 5. 输出投影
        output = np.dot(concatenated, self.W_o)
        if self.use_bias:
            output += self.b_o
        
        if return_attention:
            attention_weights = np.stack(attention_weights_list, axis=0)
            return output, attention_weights
        
        return output


class MultiHeadSelfAttention(MultiHeadAttention):
    """多头自注意力（简化接口）"""
    
    def forward(self, x, mask=None, return_attention=False):
        return super().forward(x, x, x, mask=mask, return_attention=return_attention)

## 3. 测试多头自注意力

In [ ]:
# 参数设置
seq_len = 10
embed_dim = 64
num_heads = 8

print("配置:")
print(f"  序列长度: {seq_len}")
print(f"  嵌入维度: {embed_dim}")
print(f"  注意力头数: {num_heads}")
print(f"  每个头的维度: {embed_dim // num_heads}")

# 生成输入
x = np.random.randn(seq_len, embed_dim)

# 创建多头自注意力层
mha = MultiHeadSelfAttention(embed_dim, num_heads)

# 前向传播
output, attn_weights = mha.forward(x, return_attention=True)

print(f"\n形状:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  注意力权重: {attn_weights.shape}")
print(f"    - {num_heads}个头，每个头的权重矩阵: ({seq_len}, {seq_len})")

## 4. 可视化不同头的注意力模式

In [ ]:
# 可视化前4个头的注意力权重
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    sns.heatmap(attn_weights[i], annot=True, fmt='.2f', cmap='YlOrRd', 
                ax=axes[i], cbar=True, square=True)
    axes[i].set_title(f'Head {i+1} 注意力权重')
    axes[i].set_xlabel('Key位置')
    axes[i].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("\n观察：")
print("- 每个头学习到不同的注意力模式")
print("- 有些头关注局部（对角线），有些关注全局")
print("- 不同的头捕获不同类型的依赖关系")

## 5. 因果Mask（GPT风格）

In [ ]:
def create_causal_mask(seq_len):
    """创建因果mask（下三角矩阵）"""
    return np.tril(np.ones((seq_len, seq_len)))

# 创建因果mask
causal_mask = create_causal_mask(seq_len)

print("因果Mask:")
print(causal_mask.astype(int))

# 应用mask
output_masked, attn_weights_masked = mha.forward(x, mask=causal_mask, return_attention=True)

# 可视化mask效果（第1个头）
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 原始mask
sns.heatmap(causal_mask, annot=True, fmt='.0f', cmap='Greys', 
            ax=axes[0], cbar=False, square=True)
axes[0].set_title('因果Mask')
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

# 无mask的注意力
sns.heatmap(attn_weights[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[1], square=True)
axes[1].set_title('Head 1 (无Mask)')
axes[1].set_xlabel('Key位置')
axes[1].set_ylabel('Query位置')

# 有mask的注意力
sns.heatmap(attn_weights_masked[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[2], square=True)
axes[2].set_title('Head 1 (带因果Mask)')
axes[2].set_xlabel('Key位置')
axes[2].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("\n因果Mask效果：")
print("- 每个位置只能看到自己和之前的位置")
print("- 右上角（未来信息）被屏蔽")
print("- 用于GPT等自回归模型")

## 6. 跨注意力（Cross-Attention）

In [ ]:
# Encoder-Decoder场景
encoder_seq_len = 12
decoder_seq_len = 8

encoder_output = np.random.randn(encoder_seq_len, embed_dim)
decoder_input = np.random.randn(decoder_seq_len, embed_dim)

# 创建跨注意力层
cross_attn = MultiHeadAttention(embed_dim, num_heads)

# Query来自decoder，Key和Value来自encoder
cross_output, cross_attn_weights = cross_attn.forward(
    query=decoder_input,
    key=encoder_output,
    value=encoder_output,
    return_attention=True
)

print("跨注意力:")
print(f"  Encoder输出 (Key/Value): {encoder_output.shape}")
print(f"  Decoder输入 (Query): {decoder_input.shape}")
print(f"  Cross-Attention输出: {cross_output.shape}")
print(f"  Cross-Attention权重: {cross_attn_weights.shape}")

# 可视化跨注意力（第1个头）
plt.figure(figsize=(10, 6))
sns.heatmap(cross_attn_weights[0], annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Cross-Attention权重 (Head 1)')
plt.xlabel('Encoder位置 (Key)')
plt.ylabel('Decoder位置 (Query)')
plt.show()

print("\n跨注意力说明：")
print(f"- Decoder的{decoder_seq_len}个位置关注Encoder的{encoder_seq_len}个位置")
print("- 用于Transformer的Decoder中")
print("- 实现序列到序列的对齐")

## 7. 多头 vs 单头对比

In [ ]:
# 单头注意力
single_head = MultiHeadSelfAttention(embed_dim, num_heads=1)
output_single, attn_single = single_head.forward(x, return_attention=True)

# 多头注意力（8头）
multi_head = MultiHeadSelfAttention(embed_dim, num_heads=8)
output_multi, attn_multi = multi_head.forward(x, return_attention=True)

# 对比
print("单头 vs 多头对比:\n")
print("单头注意力:")
print(f"  头数: 1")
print(f"  每个头维度: {embed_dim}")
print(f"  注意力权重: {attn_single.shape}")

print("\n多头注意力:")
print(f"  头数: 8")
print(f"  每个头维度: {embed_dim // 8}")
print(f"  注意力权重: {attn_multi.shape}")

# 计算参数量
params = embed_dim * embed_dim * 4  # W_q, W_k, W_v, W_o
print(f"\n参数量（两者相同）: {params:,}")

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(attn_single[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[0], square=True)
axes[0].set_title('单头注意力')
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

# 多头的平均注意力
attn_multi_avg = np.mean(attn_multi, axis=0)
sns.heatmap(attn_multi_avg, annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[1], square=True)
axes[1].set_title('多头注意力（平均）')
axes[1].set_xlabel('Key位置')
axes[1].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

## 8. 分析不同头的关注模式

In [ ]:
# 选择一个Query位置，观察不同头的关注模式
query_pos = 5

plt.figure(figsize=(14, 6))

for i in range(num_heads):
    attention_at_pos = attn_weights[i, query_pos, :]
    plt.plot(attention_at_pos, marker='o', label=f'Head {i+1}', alpha=0.7)

plt.xlabel('Key位置', fontsize=12)
plt.ylabel('注意力权重', fontsize=12)
plt.title(f'Query位置{query_pos}在不同头中的注意力分布', fontsize=14)
plt.legend(ncol=4, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nQuery位置{query_pos}的分析:")
for i in range(num_heads):
    most_attended = np.argmax(attn_weights[i, query_pos, :])
    print(f"  Head {i+1}: 最关注位置{most_attended} (权重={attn_weights[i, query_pos, most_attended]:.3f})")

## 9. 计算复杂度分析

In [ ]:
import time

# 测试不同序列长度的计算时间
seq_lengths = [10, 20, 50, 100, 200]
times = []

for n in seq_lengths:
    x_test = np.random.randn(n, embed_dim)
    mha_test = MultiHeadSelfAttention(embed_dim, num_heads)
    
    start = time.time()
    for _ in range(10):  # 多次运行取平均
        _ = mha_test.forward(x_test)
    elapsed = (time.time() - start) / 10
    times.append(elapsed * 1000)  # 转换为毫秒

# 可视化
plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, times, marker='o', linewidth=2, markersize=8)
plt.xlabel('序列长度 (n)', fontsize=12)
plt.ylabel('计算时间 (ms)', fontsize=12)
plt.title('多头注意力的时间复杂度', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("时间复杂度分析:")
for n, t in zip(seq_lengths, times):
    print(f"  n={n:3d}: {t:.2f} ms")

print("\n复杂度说明:")
print("  投影: O(n·d²)")
print("  注意力计算: O(n²·d)")
print("  总计: O(n²·d + n·d²)")
print("  当n较大时，主导项是O(n²·d)")

## 10. 实际应用示例

In [ ]:
print("多头注意力的实际应用:\n")

print("1. BERT (双向编码器):")
print("   - 使用多头自注意力")
print("   - 无mask，每个位置可以看到所有位置")
print("   - 用于文本理解任务")
print("   - 配置: embed_dim=768, num_heads=12\n")

print("2. GPT (单向解码器):")
print("   - 使用带因果mask的多头自注意力")
print("   - 每个位置只能看到之前的位置")
print("   - 用于文本生成任务")
print("   - 配置: embed_dim=768, num_heads=12\n")

print("3. Transformer Encoder-Decoder:")
print("   - Encoder: 多头自注意力（无mask）")
print("   - Decoder: 多头自注意力（因果mask）+ 多头跨注意力")
print("   - 用于机器翻译、摘要等任务")
print("   - 配置: embed_dim=512, num_heads=8\n")

# 模拟BERT配置
print("\n示例：BERT-Base配置")
bert_embed_dim = 768
bert_num_heads = 12
bert_seq_len = 512

bert_mha = MultiHeadSelfAttention(bert_embed_dim, bert_num_heads)
print(f"  嵌入维度: {bert_embed_dim}")
print(f"  注意力头数: {bert_num_heads}")
print(f"  每个头维度: {bert_embed_dim // bert_num_heads}")
print(f"  最大序列长度: {bert_seq_len}")

bert_params = bert_embed_dim * bert_embed_dim * 4
print(f"  单层参数量: {bert_params:,}")
print(f"  12层总参数量: {bert_params * 12:,}")

## 总结

### 多头注意力的关键特性：

1. **并行计算**：多个头同时工作，捕获不同的模式
2. **参数高效**：与单头参数量相同，表达能力更强
3. **多样性**：不同头学习不同的注意力模式
4. **灵活性**：可用于自注意力和跨注意力

### 核心公式：
$$
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O
$$

### 应用：
- BERT、GPT、T5等所有Transformer架构
- 自然语言处理的核心组件
- 跨模态注意力（视觉、语音等）